# Transformer Models
This notebook introduces the Transformer architecture and demonstrates using an encoder-only model (BERT-like),
a decoder-only model (GPT-style), and an encoder-decoder model (T5-style). We'll:

1. Load each model from Hugging Face Hub.

2. Tokenize sample text.

3. Run a forward pass and inspect outputs.

Note: keep batch sizes tiny so this runs on CPU.

In [1]:
!pip install -q transformers torch==2.8.0 torchvision==0.23.0+cu126 --index-url https://download.pytorch.org/whl/cu121 --upgrade

In [3]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, AutoModelForSeq2SeqLM
import torch

text = "Transformers are powerful models for NLP."

# Encoder-only (BERT-like)
enc_name = "distilbert/distilbert-base-uncased"
enc_tok = AutoTokenizer.from_pretrained(enc_name)
enc_model = AutoModel.from_pretrained(enc_name)
enc_inputs = enc_tok(text, return_tensors="pt")
with torch.no_grad():
    enc_outputs = enc_model(**enc_inputs)
print('Encoder hidden states shape:', enc_outputs.last_hidden_state.shape)

# Decoder-only (GPT-style)
dec_name = "gpt2"
dec_tok = AutoTokenizer.from_pretrained(dec_name)
dec_tok.pad_token = dec_tok.eos_token
dec_model = AutoModelForCausalLM.from_pretrained(dec_name)
dec_inputs = dec_tok(text, return_tensors="pt")
with torch.no_grad():
    dec_logits = dec_model(**dec_inputs).logits
print('Decoder logits shape:', dec_logits.shape)

# Encoder-decoder (T5-style)
seq2seq_name = "google/flan-t5-small"
seq_tok = AutoTokenizer.from_pretrained(seq2seq_name)
seq_model = AutoModelForSeq2SeqLM.from_pretrained(seq2seq_name)

prompt = "Translate to German: Transformers are powerful models for NLP."
inputs = seq_tok(prompt, return_tensors="pt")
with torch.no_grad():
    generated_ids = seq_model.generate(**inputs, max_length=40)
print('Seq2Seq output:', seq_tok.decode(generated_ids[0], skip_special_tokens=True))

Encoder hidden states shape: torch.Size([1, 10, 768])
Decoder logits shape: torch.Size([1, 9, 50257])
Seq2Seq output: Transformers sind leistungsfähige Modelle für NLP.
